# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors (FAIR²) Exploration with `mlcroissant`

This notebook demonstrates how to load, inspect, and analyze the [FAIR² dataset](https://sen.science/doi/10.71728/senscience.qs2f-h81p) using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. All data entities (record sets, fields, columns) are referenced by their `@id` for clarity and reproducibility.

### Dataset Source
The dataset is described using a [Croissant schema](https://mlcommons.org/croissant/) and is accessed via the following URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Install mlcroissant if not yet installed
!pip install -q mlcroissant

## 1. Data Loading

We load the dataset metadata and contents using `mlcroissant`. The documentation and downloaders abstract access to the data and its schema.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant Schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print("Dataset title: ", metadata.name)
print("Description: ", metadata.description)
print("Version: ", metadata.version)
print("License: ", metadata.license)
print("Available record sets:")
if hasattr(metadata, 'record_sets') and metadata.record_sets:
    for rs in metadata.record_sets:
        print(f"  - @id: {rs.id} | name: {getattr(rs, 'name', None)}")
else:
    print("  No record sets discovered in root metadata, will inspect via dataset API.")

## 2. Data Overview

Review the available record sets, their fields, and their `@id`s. This helps in identifying the structure and contents for further steps.

In [ ]:
# List all record sets in the dataset
print("\nRecord sets available (by @id):")
record_set_ids = []
for rs in dataset.record_sets():
    print(f"- @id: {rs.id} | Name: {getattr(rs, 'name', None)}")
    record_set_ids.append(rs.id)
    # Show a preview of fields for illustration
    if hasattr(rs, 'fields'):
        print("  Fields:")
        for field in rs.fields:
            print(f"    - @id: {field.id} | name: {getattr(field, 'name', None)} | dataType: {getattr(field, 'data_type', None)}")
    else:
        print("  (No fields found)")

if len(record_set_ids) == 0:
    print("No record sets found!")

## 3. Data Extraction

We extract data from the main record set(s) into pandas DataFrames for analysis. All record set and field references use their `@id` values as demonstrated above.

In [ ]:
# Select the primary record set for tabular analysis
# In this dataset, there may be only one main record set containing clinicopathological data. We'll use the first discovered for demonstration.
if len(record_set_ids) > 0:
    primary_record_set_id = record_set_ids[0]
    print(f"\nLoading records for record set @id: {primary_record_set_id}")

    records_iter = dataset.records(record_set=primary_record_set_id)
    records = list(records_iter)
    data_df = pd.DataFrame(records)
    print(f"Loaded {data_df.shape[0]} rows with columns:")
    pprint.pprint(list(data_df.columns))

    # Display a few rows
    data_df.head() 
else:
    print("No record sets to load!")

## 4. Exploratory Data Analysis (EDA)

Let's perform some basic EDA by filtering, normalizing numeric fields, and grouping by a clinically relevant attribute.

*All columns below are referenced by their Croissant schema `@id`.*

In [ ]:
# Identify numeric fields by scanning the DataFrame's columns & types
numeric_columns = data_df.select_dtypes(include=['number']).columns.tolist()
print(f"Numeric fields available: {numeric_columns}")

# For demonstration, let's assume 'cr:age_at_second_crc' is the @id for age at diagnosis
# If not present, substitute with the first numeric field found
numeric_field = 'cr:age_at_second_crc' if 'cr:age_at_second_crc' in data_df.columns else (numeric_columns[0] if numeric_columns else None)
print(f"Choosing {numeric_field} for numeric analysis.")

# Set a threshold for filtering
threshold = 60  # e.g., age > 60
if numeric_field:
    filtered_df = data_df[data_df[numeric_field] > threshold].copy()
    print(f"Filtered records where {numeric_field} > {threshold}:")
    print(filtered_df[[numeric_field]].head())

    # Normalize the field
    filtered_df[f"{numeric_field}_zscore"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean())/filtered_df[numeric_field].std()
    print(f"\nNormalized (z-score) {numeric_field}:")
    print(filtered_df[[numeric_field, f"{numeric_field}_zscore"]].head())

    # Attempt to group by a categorical field (e.g., anatomical site or MSI status)
    # Let's assume 'cr:anatomical_site' or otherwise pick the first object dtype column
    group_fields = [
        fid for fid in ['cr:anatomical_site', 'cr:msi_status', 'cr:sex']
        if fid in data_df.columns
    ]
    group_field = group_fields[0] if group_fields else (data_df.select_dtypes('object').columns[0] if not data_df.select_dtypes('object').empty else None)

    if group_field:
        grouped = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"\nGrouped mean {numeric_field} by {group_field}:")
        print(grouped)
else:
    print("No numeric fields available for EDA.")

## 5. Visualization

Let's visualize the distribution of the selected numeric field (e.g., age at second CRC) and its relationship to a key grouping (e.g., anatomical site).

*If running in Colab/Jupyter, you'll see the plot directly below.*

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field:
    plt.figure(figsize=(8,5))
    sns.histplot(data_df[numeric_field].dropna(), kde=True)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # If group_field exists, show boxplot
    if group_field:
        plt.figure(figsize=(10,5))
        sns.boxplot(data=data_df, x=group_field, y=numeric_field)
        plt.title(f'{numeric_field} by {group_field}')
        plt.xticks(rotation=45)
        plt.show()
else:
    print('No numeric field found for visualization.')

## 6. Conclusion

- This notebook demonstrated using `mlcroissant` to load metadata and tabular record sets from the FAIR² dataset using only Croissant `@id` references for full traceability.
- We explored numeric and categorical fields, performed simple filtering, normalization, and group-wise aggregation to mimic standard exploratory analysis workflows.
- Visualizations provide insight into patient distribution and relationships in clinicopathological variables.

*For more details and advanced usage, see [`mlcroissant` documentation](https://croissant.mlcommons.org/docs/).*